In [0]:
%sql
CREATE EXTERNAL VOLUME IF NOT EXISTS ai_analytics.source.v1_raw
LOCATION 's3://ai-analytics-johnson-2026/raw/';

In [0]:
display(
    dbutils.fs.ls("/Volumes/ai_analytics/source/v1_raw/")
)

In [0]:
raw_messages = (
    spark.read
    .option(
        "basePath",
        "/Volumes/ai_analytics/source/v1_raw/messages/"
    )
    .json(
        "/Volumes/ai_analytics/source/v1_raw/messages/"
    )
)

print("Raw messages:", raw_messages.count())
print(raw_messages.columns)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

RAW_ROOT = "/Volumes/ai_analytics/source/v1_raw"

raw_users = (
    spark.read
    .option("basePath", f"{RAW_ROOT}/users/")
    .json(f"{RAW_ROOT}/users/")
)

raw_updates = (
    spark.read
    .option("basePath", f"{RAW_ROOT}/user_updates/")
    .json(f"{RAW_ROOT}/user_updates/")
)

print("Raw users:", raw_users.count())
print("Raw user updates:", raw_updates.count())

In [0]:
generator_users = (
    raw_users
    .select(
        "user_id",
        "email",
        "name",
        "country_code",
        "account_status",
        F.to_timestamp("signup_at").alias("signup_at"),
        "signup_source",
        F.to_timestamp("ingested_at").alias("ingested_at")
    )
)

display(generator_users.limit(10))

In [0]:
window_spec = (
    Window
    .partitionBy("user_id", "field_name")
    .orderBy(
        F.to_timestamp("updated_at").desc(),
        F.col("update_id").desc()
    )
)

latest_updates = (
    raw_updates
    .withColumn("rn", F.row_number().over(window_spec))
    .filter(F.col("rn") == 1)
)

update_state = (
    latest_updates
    .groupBy("user_id")
    .pivot(
        "field_name",
        ["email", "name", "country_code", "account_status"]
    )
    .agg(F.first("new_value"))
    .select(
        "user_id",
        F.col("email").alias("updated_email"),
        F.col("name").alias("updated_name"),
        F.col("country_code").alias("updated_country_code"),
        F.col("account_status").alias("updated_account_status")
    )
)

In [0]:
generator_user_state = (
    generator_users.alias("u")
    .join(update_state.alias("s"), "user_id", "left")
    .select(
        "user_id",

        F.coalesce(
            F.col("s.updated_email"),
            F.col("u.email")
        ).alias("email"),

        F.coalesce(
            F.col("s.updated_name"),
            F.col("u.name")
        ).alias("name"),

        F.coalesce(
            F.col("s.updated_country_code"),
            F.col("u.country_code")
        ).alias("country_code"),

        F.coalesce(
            F.col("s.updated_account_status"),
            F.col("u.account_status")
        ).alias("account_status")
    )
)

display(generator_user_state.limit(10))

In [0]:
users_count = generator_users.count()
state_count = generator_user_state.count()

print("generator_users:", users_count)
print("generator_user_state:", state_count)

assert users_count == 10000
assert state_count == 10000

assert (
    generator_user_state
    .filter(F.col("user_id").isNull())
    .count()
    == 0
)

print("PASS: user generator state validated")

In [0]:
assert not spark.catalog.tableExists(
    "ai_analytics.source.generator_users"
)

assert not spark.catalog.tableExists(
    "ai_analytics.source.generator_user_state"
)

(
    generator_users
    .writeTo("ai_analytics.source.generator_users")
    .using("delta")
    .create()
)

(
    generator_user_state
    .writeTo("ai_analytics.source.generator_user_state")
    .using("delta")
    .create()
)

print("Created generator user state tables")

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN ai_analytics.source
    """)
)

In [0]:
from pyspark.sql import functions as F

RAW_ROOT = "/Volumes/ai_analytics/source/v1_raw"


def load_latest_reference(entity):
    path = f"{RAW_ROOT}/{entity}/"

    df = (
        spark.read
        .option("basePath", path)
        .json(path)
    )

    latest_dt = (
        df
        .select(F.max("dt").alias("latest_dt"))
        .first()["latest_dt"]
    )

    latest_df = (
        df
        .filter(F.col("dt") == latest_dt)
        .drop("dt")
    )

    print(
        f"{entity}: latest partition = {latest_dt}, "
        f"rows = {latest_df.count()}"
    )

    return latest_df


generator_models = load_latest_reference("models")
generator_devices = load_latest_reference("devices")
generator_plans = load_latest_reference("subscription_plans")

In [0]:
generator_models = (
    generator_models
    .select(
        "model_id",
        "model_name",
        "model_version",
        "provider",
        F.to_date("release_date").alias("release_date"),
        "active_flag"
    )
)

generator_devices = (
    generator_devices
    .select(
        "device_id",
        "device_type",
        "operating_system",
        "browser",
        "app_platform"
    )
)

generator_plans = (
    generator_plans
    .select(
        "plan_id",
        "plan_name",
        "monthly_price",
        "currency",
        "active_flag"
    )
)

In [0]:
print(generator_models.columns)
print(generator_devices.columns)
print(generator_plans.columns)

In [0]:
assert generator_models.count() == 8
assert generator_devices.count() == 12
assert generator_plans.count() == 5

assert generator_models.select("model_id").distinct().count() == 8
assert generator_devices.select("device_id").distinct().count() == 12
assert generator_plans.select("plan_id").distinct().count() == 5

print("PASS: reference generator state validated")

In [0]:
targets = {
    "ai_analytics.source.generator_models": generator_models,
    "ai_analytics.source.generator_devices": generator_devices,
    "ai_analytics.source.generator_subscription_plans": generator_plans,
}

for table_name, df in targets.items():
    assert not spark.catalog.tableExists(table_name), (
        f"{table_name} already exists — refusing to overwrite"
    )

    (
        df.writeTo(table_name)
        .using("delta")
        .create()
    )

    print(f"Created {table_name}")

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN ai_analytics.source
    """)
)

In [0]:
from pyspark.sql import functions as F
from datetime import timedelta

RAW_ROOT = "/Volumes/ai_analytics/source/v1_raw"

raw_messages = (
    spark.read
    .option("basePath", f"{RAW_ROOT}/messages/")
    .json(f"{RAW_ROOT}/messages/")
)

historical_max = (
    raw_messages
    .select(F.max("dt").alias("max_dt"))
    .first()["max_dt"]
)

recent_start = historical_max - timedelta(days=29)

print("Historical max:", historical_max)
print("Recent window:", recent_start, "→", historical_max)

In [0]:
from pyspark.sql import functions as F

RAW_ROOT = "/Volumes/ai_analytics/source/v1_raw"

historical_max = F.lit("2026-08-22").cast("date")
recent_start = F.lit("2026-07-24").cast("date")


raw_conversations = (
    spark.read
    .option("basePath", f"{RAW_ROOT}/conversations/")
    .json(f"{RAW_ROOT}/conversations/")
)

raw_messages = (
    spark.read
    .option("basePath", f"{RAW_ROOT}/messages/")
    .json(f"{RAW_ROOT}/messages/")
)


generator_recent_conversations = (
    raw_conversations
    .filter(
        (F.col("dt") >= recent_start)
        & (F.col("dt") <= historical_max)
    )
    .select(
        "conversation_id",
        "user_id",
        F.to_timestamp("created_at").alias("created_at"),
        F.to_timestamp("ingested_at").alias("ingested_at")
    )
)


generator_recent_messages = (
    raw_messages
    .filter(
        (F.col("dt") >= recent_start)
        & (F.col("dt") <= historical_max)
    )
    .select(
        "message_id",
        "conversation_id",
        "user_id",
        "device_id",
        "sequence_number",
        "message_text",
        F.to_timestamp("created_at").alias("created_at"),
        F.to_timestamp("ingested_at").alias("ingested_at")
    )
)

In [0]:
conversation_count = generator_recent_conversations.count()
message_count = generator_recent_messages.count()

print("Recent conversations:", conversation_count)
print("Recent messages:", message_count)

display(generator_recent_conversations.limit(5))
display(generator_recent_messages.limit(5))

In [0]:
assert conversation_count > 0
assert message_count > 0

assert (
    generator_recent_conversations
    .filter(
        (F.to_date("ingested_at") < F.lit("2026-07-24"))
        | (F.to_date("ingested_at") > F.lit("2026-08-22"))
    )
    .count()
    == 0
)

assert (
    generator_recent_messages
    .filter(
        (F.to_date("ingested_at") < F.lit("2026-07-24"))
        | (F.to_date("ingested_at") > F.lit("2026-08-22"))
    )
    .count()
    == 0
)

print("PASS: recent activity window validated")

In [0]:
targets = {
    "ai_analytics.source.generator_recent_conversations":
        generator_recent_conversations,

    "ai_analytics.source.generator_recent_messages":
        generator_recent_messages,
}

for table_name, df in targets.items():

    assert not spark.catalog.tableExists(table_name), (
        f"{table_name} already exists — refusing to overwrite"
    )

    (
        df.writeTo(table_name)
        .using("delta")
        .create()
    )

    print(f"Created {table_name}")

In [0]:
raw_subscriptions = (
    spark.read
    .option("basePath", f"{RAW_ROOT}/subscriptions/")
    .json(f"{RAW_ROOT}/subscriptions/")
)

generator_subscriptions = (
    raw_subscriptions
    .select(
        "subscription_id",
        "user_id",
        "plan_id",
        F.to_timestamp("started_at").alias("started_at"),
        F.to_timestamp("ended_at").alias("ended_at"),
        "subscription_status",
        "actual_monthly_price",
        F.to_timestamp("updated_at").alias("updated_at"),
        F.to_timestamp("ingested_at").alias("ingested_at")
    )
)

print("Subscriptions:", generator_subscriptions.count())

display(generator_subscriptions.limit(10))

In [0]:
assert generator_subscriptions.count() == 13153

assert (
    generator_subscriptions
    .select("subscription_id")
    .distinct()
    .count()
    == 13153
)

print("PASS: subscriptions validated")

In [0]:
table_name = "ai_analytics.source.generator_subscriptions"

assert not spark.catalog.tableExists(table_name), (
    f"{table_name} already exists — refusing to overwrite"
)

(
    generator_subscriptions
    .writeTo(table_name)
    .using("delta")
    .create()
)

print(f"Created {table_name}")

In [0]:
COUNT_ENTITIES = [
    "user_updates",
    "conversations",
    "messages",
    "errors",
]

count_frames = []

for entity in COUNT_ENTITIES:

    df = (
        spark.read
        .option("basePath", f"{RAW_ROOT}/{entity}/")
        .json(f"{RAW_ROOT}/{entity}/")
        .filter(
            (F.col("dt") >= F.lit("2026-07-24").cast("date"))
            & (F.col("dt") <= F.lit("2026-08-22").cast("date"))
        )
    )

    daily = (
        df
        .groupBy(F.col("dt").alias("activity_date"))
        .count()
        .withColumnRenamed("count", "row_count")
        .withColumn("entity", F.lit(entity))
        .select(
            "entity",
            F.to_date("activity_date").alias("activity_date"),
            "row_count"
        )
    )

    count_frames.append(daily)

In [0]:
from functools import reduce

generator_daily_counts = reduce(
    lambda left, right: left.unionByName(right),
    count_frames
)

display(
    generator_daily_counts
    .orderBy("activity_date", "entity")
)

In [0]:
print(
    "Daily count rows:",
    generator_daily_counts.count()
)

display(
    generator_daily_counts
    .groupBy("entity")
    .agg(
        F.min("activity_date").alias("first_date"),
        F.max("activity_date").alias("last_date"),
        F.count("*").alias("days")
    )
)

In [0]:
table_name = "ai_analytics.source.generator_daily_counts"

assert not spark.catalog.tableExists(table_name), (
    f"{table_name} already exists — refusing to overwrite"
)

(
    generator_daily_counts
    .writeTo(table_name)
    .using("delta")
    .create()
)

print(f"Created {table_name}")

In [0]:
raw_subscriptions = (
    spark.read
    .option("basePath", f"{RAW_ROOT}/subscriptions/")
    .json(f"{RAW_ROOT}/subscriptions/")
)

generator_subscriptions = (
    raw_subscriptions
    .select(
        "subscription_id",
        "user_id",
        "plan_id",
        F.to_timestamp("started_at").alias("started_at"),
        F.to_timestamp("ended_at").alias("ended_at"),
        "subscription_status",
        "actual_monthly_price",
        F.to_timestamp("updated_at").alias("updated_at"),
        F.to_timestamp("ingested_at").alias("ingested_at")
    )
)

print("Subscriptions:", generator_subscriptions.count())
display(generator_subscriptions.limit(10))

In [0]:
subscription_count = generator_subscriptions.count()
unique_subscription_count = (
    generator_subscriptions
    .select("subscription_id")
    .distinct()
    .count()
)

print("Rows:", subscription_count)
print("Unique IDs:", unique_subscription_count)

assert subscription_count == 13153
assert unique_subscription_count == 13153

print("PASS: subscriptions validated")

In [0]:
existing_subscriptions = spark.table(
    "ai_analytics.source.generator_subscriptions"
)

row_count = existing_subscriptions.count()

unique_count = (
    existing_subscriptions
    .select("subscription_id")
    .distinct()
    .count()
)

print("Rows:", row_count)
print("Unique subscription IDs:", unique_count)

assert row_count == 13153
assert unique_count == 13153

print("PASS: existing generator_subscriptions is valid")

In [0]:
from functools import reduce
from pyspark.sql import functions as F

COUNT_ENTITIES = [
    "user_updates",
    "conversations",
    "messages",
    "errors",
]

count_frames = []

for entity in COUNT_ENTITIES:
    df = (
        spark.read
        .option("basePath", f"{RAW_ROOT}/{entity}/")
        .json(f"{RAW_ROOT}/{entity}/")
        .filter(
            (F.col("dt") >= F.lit("2026-07-24").cast("date"))
            & (F.col("dt") <= F.lit("2026-08-22").cast("date"))
        )
    )

    daily = (
        df
        .groupBy(F.col("dt").alias("activity_date"))
        .count()
        .withColumnRenamed("count", "row_count")
        .withColumn("entity", F.lit(entity))
        .select(
            "entity",
            F.to_date("activity_date").alias("activity_date"),
            "row_count"
        )
    )

    count_frames.append(daily)

generator_daily_counts = reduce(
    lambda left, right: left.unionByName(right),
    count_frames
)

In [0]:
display(
    generator_daily_counts
    .orderBy("activity_date", "entity")
)

In [0]:
summary = (
    generator_daily_counts
    .groupBy("entity")
    .agg(
        F.min("activity_date").alias("first_date"),
        F.max("activity_date").alias("last_date"),
        F.count("*").alias("days")
    )
)

display(summary)

In [0]:
existing_daily_counts = spark.table(
    "ai_analytics.source.generator_daily_counts"
)

display(
    existing_daily_counts
    .groupBy("entity")
    .agg(
        F.min("activity_date").alias("first_date"),
        F.max("activity_date").alias("last_date"),
        F.count("*").alias("days")
    )
    .orderBy("entity")
)

In [0]:
print("Rows:", existing_daily_counts.count())

display(
    existing_daily_counts
    .orderBy("activity_date", "entity")
    .limit(20)
)